# ShiftLog-Gym GRPO Training (Free T4)

Three-stage memory-policy training stack for ShiftLog-Gym using Unsloth, TRL GRPO, and W&B free.

In [ ]:
# Colab setup: clone repo + install it so `shiftlog_gym` is importable.
import os

REPO_URL = "https://github.com/Chirag0096/ShiftLog-Gym.git"
REPO_DIR = "ShiftLog-Gym"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!pip -q install -e .


In [ ]:
!pip -q install unsloth trl transformers datasets peft accelerate bitsandbytes wandb huggingface_hub matplotlib pandas seaborn


In [ ]:
import json
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from datasets import Dataset
from huggingface_hub import notebook_login
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer, SFTConfig, SFTTrainer
from transformers import AutoTokenizer

from shiftlog_gym.scenarios import FAMILIES
from shiftlog_gym.simulator import ShiftLogSimulator
from shiftlog_gym.training import (
    TEST_VARIANTS,
    TRAIN_VARIANTS,
    VALID_VARIANTS,
    build_variant_split,
    scripted_policy,
    summarize_episode,
    write_artifacts,
)
from shiftlog_gym.trl_env import (
    ShiftLogToolEnv,
    reward_efficiency,
    reward_hallucination,
    reward_memory_integrity,
    reward_memory_write,
    reward_recall,
    reward_success,
)

sns.set_theme(style='whitegrid')


In [ ]:
try:
    import wandb
    WANDB_AVAILABLE = True
except Exception:
    WANDB_AVAILABLE = False

WANDB_PROJECT = 'shiftlog-gym'
WANDB_ENABLED = WANDB_AVAILABLE and bool(os.environ.get('WANDB_API_KEY'))

if WANDB_ENABLED:
    wandb.login(key=os.environ['WANDB_API_KEY'])
    print('W&B enabled')
else:
    print('W&B disabled. Set WANDB_API_KEY in Colab secrets to enable external tracking.')


In [ ]:
@dataclass
class TrainingPreset:
    model_name: str = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
    fallback_model_name: str = 'Qwen/Qwen2.5-1.5B-Instruct'
    max_seq_length: int = 2048
    max_completion_length: int = 768
    stage_b_max_completion_length: int = 384
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 4
    num_generations: int = 4
    learning_rate: float = 5e-6
    save_steps: int = 25
    logging_steps: int = 1
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.0
    seed: int = 42


cfg = TrainingPreset()
split_config = build_variant_split()
pd.DataFrame([(name, list(indices)) for name, indices in split_config.items()], columns=['split', 'variant_indices'])


In [ ]:
random.seed(cfg.seed)

def build_env_dataset(variants, families=None, repeats_per_family=2, stage='full', seed=42):
    families = families or list(FAMILIES)
    prompt = (
        'You are the primary SRE on call in ShiftLog-Gym. '\
        'Use the shift log intentionally. Before acting on repeated or causally linked incidents, retrieve memory. '\
        'Write compact structured facts. Avoid contradictions and unsupported mitigations. '
        f'Current curriculum stage: {stage}.'
    )
    rows = {'prompt': [], 'family': [], 'variant_index': [], 'seed': [], 'stage': []}
    local_random = random.Random(seed)
    for family in families:
        for variant_index in variants:
            for _ in range(repeats_per_family):
                rows['prompt'].append([{'role': 'user', 'content': prompt}])
                rows['family'].append(family)
                rows['variant_index'].append(variant_index)
                rows['seed'].append(local_random.randint(0, 10_000))
                rows['stage'].append(stage)
    return Dataset.from_dict(rows)


stage_b_dataset = build_env_dataset(
    variants=TRAIN_VARIANTS,
    families=['db_pool_exhaustion', 'auth_timeout_cascade'],
    repeats_per_family=2,
    stage='stage_b',
    seed=cfg.seed,
)
stage_c_dataset = build_env_dataset(
    variants=TRAIN_VARIANTS,
    families=list(FAMILIES),
    repeats_per_family=2,
    stage='stage_c',
    seed=cfg.seed + 1,
)
valid_dataset = build_env_dataset(
    variants=VALID_VARIANTS,
    families=list(FAMILIES),
    repeats_per_family=1,
    stage='valid',
    seed=cfg.seed + 2,
)

print(stage_b_dataset)
print(stage_c_dataset)
print(valid_dataset)


## Stage A: optional tool-use bootstrap

This is only needed if the base model fails to emit valid tool calls during evaluation.

In [ ]:
RUN_STAGE_A_BOOTSTRAP = False
BOOTSTRAP_OUTPUT_DIR = 'outputs/bootstrap-sft'

def build_bootstrap_records(limit_per_family=2):
    records = []
    for family in FAMILIES:
        for variant_index in TRAIN_VARIANTS[:limit_per_family]:
            simulator = ShiftLogSimulator()
            simulator.reset(seed=cfg.seed, family=family, variant_index=variant_index)
            while not simulator.done:
                incident = simulator.active_incident
                if incident is None:
                    break
                observation = simulator.last_observation
                if incident.required_memory_keys:
                    action = {'tool': 'read_shift_log', 'arguments': {'query': ' '.join(incident.relevant_memory_terms[:3]), 'limit': 3}}
                else:
                    first_diag = next(iter(incident.diagnostics.keys()))
                    action = {'tool': 'run_diagnostic', 'arguments': {'service': incident.service, 'diagnostic': first_diag}}
                records.append({
                    'messages': [
                        {'role': 'system', 'content': 'Emit exactly one JSON tool call.'},
                        {'role': 'user', 'content': observation},
                        {'role': 'assistant', 'content': json.dumps(action)},
                    ]
                })
                scripted_policy(simulator)
                break
    return Dataset.from_list(records)


bootstrap_dataset = build_bootstrap_records()
bootstrap_dataset


In [ ]:
USE_UNSLOTH = True

def load_policy_model(model_name, max_seq_length, use_unsloth=True):
    if use_unsloth:
        try:
            from unsloth import FastLanguageModel
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=model_name,
                max_seq_length=max_seq_length,
                load_in_4bit=True,
            )
            model = FastLanguageModel.get_peft_model(
                model,
                r=cfg.lora_r,
                lora_alpha=cfg.lora_alpha,
                lora_dropout=cfg.lora_dropout,
                bias='none',
                use_gradient_checkpointing='unsloth',
            )
            return model, tokenizer, None
        except Exception as exc:
            print('Unsloth load failed, falling back to standard HF load:', exc)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    peft_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        task_type='CAUSAL_LM',
    )
    return model_name, tokenizer, peft_config


policy_model, policy_tokenizer, peft_config = load_policy_model(cfg.model_name, cfg.max_seq_length, USE_UNSLOTH)
type(policy_model), type(policy_tokenizer)


In [ ]:
if RUN_STAGE_A_BOOTSTRAP:
    sft_args = SFTConfig(
        output_dir=BOOTSTRAP_OUTPUT_DIR,
        learning_rate=2e-5,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        logging_steps=1,
        report_to='wandb' if WANDB_ENABLED else [],
        max_seq_length=cfg.max_seq_length,
    )
    sft_trainer = SFTTrainer(
        model=policy_model,
        processing_class=policy_tokenizer,
        train_dataset=bootstrap_dataset,
        args=sft_args,
    )
    sft_trainer.train()
    sft_trainer.save_model(BOOTSTRAP_OUTPUT_DIR)
else:
    print('Stage A skipped. Enable RUN_STAGE_A_BOOTSTRAP only if tool-call validity is poor.')


## GRPO stages

Stage B uses fewer families and shorter completions to teach read-before-act behavior. Stage C trains on the full current environment.

In [ ]:
RUN_STAGE_B = False
RUN_STAGE_C = False

reward_funcs = [
    reward_success,
    reward_recall,
    reward_memory_write,
    reward_memory_integrity,
    reward_efficiency,
    reward_hallucination,
]

def build_grpo_args(output_dir, max_completion_length):
    return GRPOConfig(
        output_dir=output_dir,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        logging_steps=cfg.logging_steps,
        save_steps=cfg.save_steps,
        max_completion_length=max_completion_length,
        num_generations=cfg.num_generations,
        report_to='wandb' if WANDB_ENABLED else [],
        log_completions=True,
        seed=cfg.seed,
    )


In [ ]:
if RUN_STAGE_B:
    stage_b_args = build_grpo_args('outputs/grpo-stage-b', cfg.stage_b_max_completion_length)
    stage_b_trainer = GRPOTrainer(
        model=policy_model,
        processing_class=policy_tokenizer,
        reward_funcs=reward_funcs,
        train_dataset=stage_b_dataset,
        eval_dataset=valid_dataset,
        args=stage_b_args,
        environment_factory=ShiftLogToolEnv,
        peft_config=peft_config,
    )
    stage_b_trainer.train()
    stage_b_trainer.save_model('outputs/grpo-stage-b')
else:
    print('Stage B skipped. Turn RUN_STAGE_B = True when you want the first RL pass.')


In [ ]:
if RUN_STAGE_C:
    stage_c_args = build_grpo_args('outputs/grpo-stage-c', cfg.max_completion_length)
    stage_c_trainer = GRPOTrainer(
        model=policy_model,
        processing_class=policy_tokenizer,
        reward_funcs=reward_funcs,
        train_dataset=stage_c_dataset,
        eval_dataset=valid_dataset,
        args=stage_c_args,
        environment_factory=ShiftLogToolEnv,
        peft_config=peft_config,
    )
    stage_c_trainer.train()
    stage_c_trainer.save_model('outputs/grpo-stage-c')
else:
    print('Stage C skipped. Turn RUN_STAGE_C = True after Stage B is stable or for a direct full-pass run.')


In [ ]:
def scripted_eval_rows(split='valid', seeds=(cfg.seed,)):
    variants = build_variant_split()[split]
    rows = []
    memory_events = []
    tool_events = []
    for family in FAMILIES:
        for variant_index in variants:
            for seed in seeds:
                simulator = ShiftLogSimulator()
                simulator.reset(seed=seed, family=family, variant_index=variant_index)
                scripted_policy(simulator)
                artifacts = summarize_episode(simulator, f'scripted-{family}-{variant_index}-{seed}', split, seed, variant_index)
                rows.append(artifacts.episode_row)
                memory_events.extend(artifacts.memory_events)
                tool_events.extend(artifacts.tool_timeline)
    return rows, memory_events, tool_events


eval_rows, eval_memory, eval_tools = scripted_eval_rows(split='valid')
eval_df = pd.DataFrame(eval_rows)
eval_df[['family', 'weighted_reward', 'R_recall', 'recall_before_action_rate', 'linked_incident_success_rate', 'memory_precision', 'contradiction_rate']]


In [ ]:
artifact_dir = Path('artifacts/training')
write_artifacts(artifact_dir / 'scripted_valid', eval_rows, eval_memory, eval_tools)
print('Wrote training-stage artifacts to', artifact_dir)


## Next notebook

After a GRPO run completes, open `03_eval_publish_colab.ipynb` to compare base vs trained adapters on held-out variants and publish artifacts to Hugging Face.